# The Hidden Markov Model Environment

Before we begin the spy game, we need to build the simulation engine that drives the world.

A **Hidden Markov Model (HMM)** consists of:

- **Hidden States** — the spy's true location, which we cannot observe directly.
- **Observations** — the reports we receive, which may be accurate or misleading.
- **Transition Probabilities** — the likelihood of moving from one city to another.
- **Observation Probabilities** — the likelihood of receiving a particular report from each city.

In this cell, we define the `HiddenMarkovModelEnv` class, which serves as the game environment.

The class:

- Tracks the spy's true location.
- Simulates movement using probabilistic state transitions.
- Generates truthful or deceptive reports using the sensor model.
- Provides visualization tools to help us understand the map structure and probability distributions.

**Run the cell below to load the HMM environment into your notebook.**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

class HiddenMarkovModelEnv:
    def __init__(self, transition_matrix, sensor_matrix, initial_probs, state_names=None, observation_names=None):
        self.transition_matrix = np.array(transition_matrix)
        self.sensor_matrix = np.array(sensor_matrix)
        self.pi = np.array(initial_probs)

        self.num_states = self.transition_matrix.shape[0]
        self.num_obs = self.sensor_matrix.shape[1]

        self.states = state_names if state_names else [f"S{i}" for i in range(self.num_states)]
        self.observations = observation_names if observation_names else [f"O{i}" for i in range(self.num_obs)]

    def next_state(self, current_state):
        return np.random.choice(self.num_states, p=self.transition_matrix[current_state])

    def emit(self, state):
        return np.random.choice(self.num_obs, p=self.sensor_matrix[state])

    def plot_transition_heatmap(self):
        plt.figure(figsize=(6, 5))
        sns.heatmap(self.transition_matrix, annot=True, fmt=".2f", cmap="Blues",
                    xticklabels=self.states, yticklabels=self.states, cbar=False, square=True)
        plt.title("State Transition Probabilities", fontsize=14, fontweight='bold')
        plt.xlabel("To State")
        plt.ylabel("From State")
        plt.tight_layout()
        plt.show()

    def plot_transition_graph(self):
        G = nx.DiGraph()
        for i in range(self.num_states):
            for j in range(self.num_states):
                if self.transition_matrix[i, j] > 0:
                    G.add_edge(self.states[i], self.states[j], weight=self.transition_matrix[i, j])

        plt.figure(figsize=(6, 6))
        pos = nx.circular_layout(G) 
        nx.draw_networkx_nodes(G, pos, node_size=2000, node_color="skyblue", edgecolors='black')
        nx.draw_networkx_labels(G, pos, font_size=12, font_weight="bold")
        
        ax = plt.gca()
        for edge in G.edges(data=True):
            source, target, data = edge
            if source != target:
                rad = 0.2
                ax.annotate("", xy=pos[target], xytext=pos[source],
                            arrowprops=dict(arrowstyle="->", color="gray", connectionstyle=f"arc3,rad={rad}", lw=2))
                mid_x, mid_y = (pos[source][0] + pos[target][0]) / 2, (pos[source][1] + pos[target][1]) / 2
                offset_x, offset_y = (pos[target][1] - pos[source][1]) * 0.1, (pos[source][0] - pos[target][0]) * 0.1
                txt_pos = (mid_x + offset_x, mid_y + offset_y)
            else:
                rad = 0.5
                ax.annotate("", xy=pos[source], xytext=pos[source],
                            arrowprops=dict(arrowstyle="->", color="gray", connectionstyle=f"arc3,rad={rad}", lw=2))
                txt_pos = (pos[source][0] * 1.3, pos[source][1] * 1.3)
            
            plt.text(txt_pos[0], txt_pos[1], f"{data['weight']:.2f}", fontsize=10, color="darkred", weight="bold", ha='center', va='center')

        plt.title("Two Spies Map Transition Network", fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.xlim(-1.5, 1.5)
        plt.ylim(-1.5, 1.5)
        plt.tight_layout()
        plt.show()

# Encoding the Map Rules into Matrices

To use Bayesian inference, we must convert the game's movement and reporting rules into mathematical form.

## 1. Transition Matrix ($T$)

The transition matrix describes how the spy moves between cities.

- Each **row** represents the current city.
- Each **column** represents the next city.
- Each entry contains the probability of moving from one city to another.

For example:

- From **City 1**, a roll of `1–2` moves the spy to **City 2** with probability $2/6$.
- A roll of `3–6` moves the spy to **City 3** with probability $4/6$.

## 2. Sensor Matrix ($S$)

The sensor matrix describes the reports received by the Hunter.

- Each **row** represents the spy's true city.
- Each **column** represents a reported zone (`Left`, `Center`, or `Right`).
- Each entry contains the probability of receiving that report.

Spies tell the truth on rolls of `3–6` ($4/6$ probability). When required to lie, they may report a different zone according to the game rules. These rules determine the observation probabilities stored in the sensor matrix.

**Complete the last row in each matrix. Then, run the cell to create the HMM environment and visualize the resulting transition and sensor models.**

In [ ]:
# Map 1 Data Layout
transitions1 = np.array([
    [0.0, 2/6, 4/6, 0.0, 0.0, 0.0],  # State 1
    [2/6, 0.0, 4/6, 0.0, 0.0, 0.0],  # State 2
    [0.0, 1/6, 0.0, 3/6, 2/6, 0.0],  # State 3
    [0.0, 0.0, 3/6, 0.0, 2/6, 1/6],  # State 4
    [0.0, 0.0, 0.0, 2/6, 0.0, 4/6],  # State 5
    []   # State 6
])

sensor1 = np.array(
    [
        [2/3, 1/6, 1/6],
        [2/3, 1/6, 1/6],
        [1/6, 2/3, 1/6],
        [1/6, 2/3, 1/6],
        [1/6, 1/6, 2/3],
        []
    ]
)

spy_env1 = HiddenMarkovModelEnv(
    transition_matrix=transitions1,
    sensor_matrix=sensor1,
    initial_probs=[1/6]*6,
    state_names=['1', '2', '3', '4', '5', '6'],
    observation_names=['Left', 'Center', 'Right']
)

spy_env1.plot_transition_heatmap()
spy_env1.plot_transition_graph()

# Bayesian Inference — Step-by-Step Calculations

Before relying on a computer simulation, let's examine the mathematics behind Bayesian filtering.

Our goal is to maintain a **belief distribution**, a probability vector describing how likely the spy is to be in each city.

At every time step, we repeat two operations:

## 1. Predict

Project the current belief state forward using the transition matrix:

$$
\bar{P}(C_t) = P(C_{t-1})T
$$

This step answers the question:

> "What city could the spy be in now, based only on how I know they move?"

## 2. Update

Incorporate the latest observation using Bayes' Rule:

$$
P(C_t \mid O_t) \propto \bar{P}(C_t)\odot S[:, O_t]
$$

where:

- $\odot$ denotes element-wise multiplication.
- $S[:, O_t]$ is the sensor-model column corresponding to the observed report.

Finally, normalize the resulting vector so that all probabilities sum to 1.

This step answers the question:

> "Given the report we just received, how should our beliefs change?"

**Let's code these operations together and see how the belief changes at each timestep.**

In [ ]:
obs_map = {'Left': 0, 'Center': 1, 'Right': 2}

### Round 1

# initial believ

# predict new location

# update belief with observation

# normalize

# Analyze Your Match

Now it's time to compare human decision-making with optimal Bayesian inference.

Use data from a game your team has already played and see how well the mathematical model tracks the spy.

### Instructions

Update the following lists in the code cell below:

- `recorded_observations` — Deep Cover reports from **T = 1**
- `actual_spy_locations` — Deep Cover locations from **T = 0**
- `hunter_spy_guesses` — Hunter guesses from **T = 1**

```python
recorded_observations = ['Left', 'Center', 'Right']
actual_spy_locations = [1, 3, 4, 2, 5, 6]
hunter_spy_guesses = [3, 4, 2, 5, 6]
```

After updating the lists, run the cell to generate a complete analysis of your game.

### Interpreting the Visualization
- Blue circles show the Deep Cover Spy's actual location.
- Red X's show the Hunter Spy's guesses.
- The background heatmap represents the Bayesian belief state:
  - Darker regions indicate higher probability.
  - Lighter regions indicate lower probability.

**Consider:**
- Where did the deceptive reports cause the Hunter's beliefs to diverge?
- Where does the Bayesian inference predict incorrectly? Why?

In [8]:
def plot_student_game_analysis(transitions, sensor, state_names, obs_names, observed_sequence, true_state_sequence=None, student_guesses=None):
    obs_indices = [obs_names.index(o) for o in observed_sequence]
    
    # T=0: Initial uniform belief before any observations
    current_belief = np.array([1/6] * 6)
    beliefs_over_time = [current_belief.copy()]
    
    # Step through observations (T=1, T=2, ...)
    for obs in obs_indices:
        predicted = current_belief.dot(transitions)
        unnormalized = predicted * sensor[:, obs]
        current_belief = unnormalized / np.sum(unnormalized)
        beliefs_over_time.append(current_belief.copy())
        
    beliefs_matrix = np.array(beliefs_over_time).T
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    sns.heatmap(beliefs_matrix, annot=True, fmt=".2f", cmap="YlGnBu", 
                xticklabels=[f"T={i}" for i in range(len(beliefs_over_time))],
                yticklabels=state_names, cbar_kws={'label': 'Mathematical Probability'}, ax=ax1)
    
    # True spy sequence contains the T=0 start state, so it aligns perfectly with t = 0, 1, 2...
    if true_state_sequence:
        for t, state in enumerate(true_state_sequence):
            ax1.plot(t, state_names.index(str(state)) + 0.5, marker='o', color='blue', 
                     markersize=14, markeredgewidth=3, markeredgecolor='blue', label='True Spy' if t==0 else "")
            
    # Student guesses start at T=1, so we add 1 to the plot index (t + 1)
    if student_guesses:
        for t, guess in enumerate(student_guesses):
            ax1.plot(t + 1, state_names.index(str(guess)) + 0.5, marker='x', color='crimson', 
                     markersize=12, markeredgewidth=4, label='Hunter Guess' if t==0 else "")

    ax1.set_title("Spy Hunt Analysis: Beliefs & Observations", fontsize=14, fontweight='bold', pad=40)
    ax1.set_xlabel("Game Timestep", fontsize=12)
    ax1.set_ylabel("City (Hidden State)", fontsize=12)
    
    ax2 = ax1.twiny()
    ax2.set_xlim(ax1.get_xlim())
    ax2.set_xticks(np.arange(len(beliefs_over_time)) + 0.5)
    ax2.set_xticklabels(["[Start]"] + observed_sequence, fontsize=11, fontweight='bold', color='darkblue')
    ax2.set_xlabel("Spy Report Received (Observation)", fontsize=12, color='darkblue', labelpad=10)
    
    if true_state_sequence or student_guesses:
        ax1.legend(loc='upper left', bbox_to_anchor=(1.25, 1.0))
    plt.tight_layout()
    plt.show()

In [ ]:
recorded_observations = []
actual_spy_locations  = []
hunter_spy_guesses    = []

plot_student_game_analysis(transitions1, sensor1, spy_env1.states, spy_env1.observations, recorded_observations, actual_spy_locations, hunter_spy_guesses)

# The Second Map

Apply everything you've learned to analyse the games on the other map.

Your task is to:

1. Create a new transition matrix.
2. Create a new sensor matrix.
3. Initialize a new `HiddenMarkovModelEnv`.
4. Compare your own reasoning with the optimal Bayesian belief updates.

As you work through the second map, consider:

- Which cities become difficult to distinguish?
- Which observations provide the most useful information?
- How quickly does the Bayesian model converge on the spy's location?
- How does your intuition compare with the probabilistic model?

Use the visualizations and belief distributions to evaluate your strategy and improve your predictions.

In [ ]:
# Map 2 Data Layout
transitions2 = np.array()

sensor2 = np.array()

spy_env2 = HiddenMarkovModelEnv()

spy_env2.plot_transition_heatmap()
spy_env2.plot_transition_graph()

In [ ]:
recorded_observations = []
actual_spy_locations  = []
hunter_spy_guesses    = []

plot_student_game_analysis(transitions2, sensor2, spy_env2.states, spy_env2.observations, recorded_observations, actual_spy_locations, hunter_spy_guesses)